# Movie Search with native Valkey GLIDE

Use the Python 3.13 uv environment belonging to this project. The API preserves bytes for FLOAT32 vectors; decode display text only.


In [ ]:
from rich.console import Console
from valkey_client_manager import ValkeyClientManager
from search_demo import prepare, display, document_keys, PREFIX, INDEX

manager = ValkeyClientManager(default_port=16379)
client = manager.client
Console().print(client.ping().decode())

## Model and index movies
`HSET` stores fields. `FT.CREATE` defines TEXT, TAG, NUMERIC, and HNSW vector fields. This synthetic corpus uses three-dimensional genre vectors.


In [ ]:
prepare(client)

## Instant search and title suggestions
`FT.SEARCH` handles text queries and prefix searches.


In [ ]:
display(
    client,
    document_keys(client.custom_command(["FT.SEARCH", INDEX, "orbit", "NOCONTENT"])),
    "Title search",
)
display(
    client,
    document_keys(
        client.custom_command(["FT.SEARCH", INDEX, "@title:(Lu*)", "NOCONTENT"])
    ),
    "Title suggestions",
)

## Because you watched
Binary-safe KNN parameters retrieve similar vectors. Exclude the watched title before displaying recommendations.


In [ ]:
import numpy as np
from valkey_client_manager import search_document_keys

keys = search_document_keys(
    client, INDEX, np.asarray([1, 0, 0], dtype=np.float32).tobytes(), 3
)
display(
    client,
    [key for key in keys if key != (PREFIX + "1").encode()],
    "Because you watched Orbit Rescue",
)

## Trending queries
SORTBY uses the indexed numeric popularity field; LIMIT bounds the result.


In [ ]:
display(
    client,
    document_keys(
        client.custom_command(
            [
                "FT.SEARCH",
                INDEX,
                "*",
                "NOCONTENT",
                "SORTBY",
                "popularity",
                "DESC",
                "LIMIT",
                "0",
                "3",
            ]
        )
    ),
    "Trending",
)

## Cleanup the connection
Call `close()` when finished; no global database flush is used.


In [ ]:
manager.close()